# Asset Evaluation and Portfolio Diversification Research

Production Colab launcher for `hh4832/asset_selection`. Core data loading and analysis remain in `src/`; this notebook only prepares credentials and launches `app.py`.

In [ ]:
%cd /content
!rm -rf /content/asset_selection
!git clone https://github.com/hh4832/asset_selection.git
%cd /content/asset_selection
!git log -1 --oneline
!git status --short

In [ ]:
!pip install -q -r requirements.txt
import platform, pandas, numpy, streamlit
print('Python:', platform.python_version())
print('pandas:', pandas.__version__)
print('numpy:', numpy.__version__)
print('streamlit:', streamlit.__version__)

## Credentials
Add `TIINGO_API_TOKEN` for US analysis. For headless Taiwan analysis, add `FINLAB_REFRESH_TOKEN`, `FINLAB_SESSION_ID`, and `FINLAB_API_KEY`; alternatively enable the supported interactive `finlab.login()` cell below. Secrets are copied only into runtime environment variables and are never printed.

In [ ]:
import os
from google.colab import userdata

credential_status = {}
secret_names = ['TIINGO_API_TOKEN', 'FINLAB_REFRESH_TOKEN', 'FINLAB_SESSION_ID', 'FINLAB_API_KEY']
for secret_name in secret_names:
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None
    if secret_value:
        os.environ[secret_name] = secret_value
        credential_status[secret_name] = 'loaded'
    else:
        credential_status[secret_name] = 'not configured'
print(credential_status)
print('Use Tiingo for US, or the supported FinLab login flow for TW.')

In [ ]:
# Set True only when you want the supported interactive FinLab browser login for TW analysis.
FINLAB_INTERACTIVE_LOGIN = False
if FINLAB_INTERACTIVE_LOGIN:
    import finlab
    finlab.login()
    print('FinLab authentication completed.')

In [ ]:
!pytest -q

In [ ]:
import subprocess, time, requests
from pathlib import Path

streamlit_log = open('/content/asset_selection_streamlit.log', 'w')
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.headless=true', '--server.port=8501'],
    stdout=streamlit_log, stderr=subprocess.STDOUT, cwd='/content/asset_selection'
)
for _ in range(30):
    try:
        if requests.get('http://127.0.0.1:8501/_stcore/health', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError(Path('/content/asset_selection_streamlit.log').read_text()[-4000:])
print('Streamlit started successfully.')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import re, subprocess, time

tunnel_process = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
public_url = None
for _ in range(120):
    line = tunnel_process.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.25)
if not public_url:
    raise RuntimeError('Cloudflare tunnel did not return a public URL.')
print('Asset Selection Streamlit URL:', public_url)